In [ ]:
!pip install chess

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 12.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for chess: filename=chess-1.11.2-py3-none-any.whl size=147775 sha256=b27eb4e8a328614780dbcdbace10b5c642809a644377ffb85b7e36e52246c832
  Stored in directory: /root/.cache/pip/wheels/fb/5d/5c/59a62d8a695285e59ec9c1f66add6f8a9ac4152499a2be0113
Successfully built chess


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math
import chess
import random
#import time
#import os
#import gc
import torch
import torch.nn as nn
import torch.nn.functional as F
#from torch.utils.data import DataLoader, TensorDataset
#from torch.cuda.amp import autocast, GradScaler
from tqdm.notebook import trange
torch.manual_seed(0)

In [ ]:
def load_checkpoint(checkpoint_path, model, optimizer=None, device='cpu'):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    if optimizer is not None and 'optimizer_state_dict' in checkpoint:
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    start_epoch = checkpoint.get('epoch', 0)
    print(f"Loaded checkpoint from epoch {start_epoch}")
    return start_epoch

def save_checkpoint(model, optimizer, epoch, checkpoint_dir="checkpoints"):
    if not os.path.exists(checkpoint_dir):
        os.makedirs(checkpoint_dir)
    checkpoint_path = os.path.join(checkpoint_dir, f"model_epoch_{epoch}.pt")
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
    }, checkpoint_path)
    print(f"Checkpoint saved at {checkpoint_path}")

def train_with_dataloader(model, optimizer, memory, args, device):
    # memory is a list of tuples: (encoded_state, policy_target, value_target)
    # Convert the memory to numpy arrays
    states, policy_targets, value_targets = zip(*memory)
    states_np = np.array(states)              # Expected shape: (num_samples, channels, height, width)
    policy_targets_np = np.array(policy_targets)  # Expected shape: (num_samples, action_size)
    value_targets_np = np.array(value_targets).reshape(-1, 1)  # Expected shape: (num_samples, 1)

    # Convert numpy arrays to torch tensors
    states_tensor = torch.tensor(states_np, dtype=torch.float32)
    policy_tensor = torch.tensor(policy_targets_np, dtype=torch.float32)
    value_tensor = torch.tensor(value_targets_np, dtype=torch.float32)

    # Create a TensorDataset and DataLoader
    dataset = TensorDataset(states_tensor, policy_tensor, value_tensor)
    dataloader = DataLoader(dataset,
                            batch_size=args['batch_size'],
                            shuffle=True,
                            num_workers=3,       # Use more workers if your CPU can handle it. Default: 2
                            pin_memory=True)     # Enables faster data transfer to GPU

    model.train()
    for epoch in range(args['num_epochs']):
        epoch_loss = 0
        for state_batch, policy_target_batch, value_target_batch in dataloader:
            state_batch = state_batch.to(device, non_blocking=True)
            policy_target_batch = policy_target_batch.to(device, non_blocking=True)
            value_target_batch = value_target_batch.to(device, non_blocking=True)

            optimizer.zero_grad()
            out_policy, out_value = model(state_batch)
            log_policy = F.log_softmax(out_policy, dim=1)
            policy_loss = -torch.mean(torch.sum(policy_target_batch * log_policy, dim=1))
            value_loss = F.mse_loss(out_value, value_target_batch)
            loss = policy_loss + value_loss
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        print(f"Epoch {epoch} loss: {epoch_loss/len(dataloader)}")

def train_with_dataloader_amp(model, optimizer, memory, args, device):
    # Unpack memory into numpy arrays
    states, policy_targets, value_targets = zip(*memory)
    states_np = np.array(states)
    policy_targets_np = np.array(policy_targets)
    value_targets_np = np.array(value_targets).reshape(-1, 1)

    # Convert to tensors
    states_tensor = torch.tensor(states_np, dtype=torch.float32)
    policy_tensor = torch.tensor(policy_targets_np, dtype=torch.float32)
    value_tensor = torch.tensor(value_targets_np, dtype=torch.float32)

    # Create DataLoader
    dataset = TensorDataset(states_tensor, policy_tensor, value_tensor)
    dataloader = DataLoader(dataset, batch_size=args['batch_size'], shuffle=True, num_workers=2, pin_memory=True)

    model.train()
    scaler = torch.amp.GradScaler("cuda" if torch.cuda.is_available() else "cpu")  # Initialize AMP Gradient Scaler

    for epoch in range(args['num_epochs']):
        epoch_loss = 0
        for state_batch, policy_target_batch, value_target_batch in dataloader:
            state_batch = state_batch.to(device, non_blocking=True)
            policy_target_batch = policy_target_batch.to(device, non_blocking=True)
            value_target_batch = value_target_batch.to(device, non_blocking=True)

            optimizer.zero_grad()

            # Enable Mixed Precision Training
            with torch.amp.autocast("cuda" if torch.cuda.is_available() else "cpu"):
                out_policy, out_value = model(state_batch)
                log_policy = F.log_softmax(out_policy, dim=1)
                policy_loss = -torch.mean(torch.sum(policy_target_batch * log_policy, dim=1))
                value_loss = F.mse_loss(out_value, value_target_batch)
                loss = policy_loss + value_loss

            # Scale the loss and backpropagate
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            epoch_loss += loss.item()

        print(f"Epoch {epoch} loss: {epoch_loss/len(dataloader)}")

        # Save a checkpoint after each epoch
        #save_checkpoint(model, optimizer, epoch)
        # Clear the GPU cache to free up memory
        #torch.cuda.empty_cache()

# **AlphaZero AI Model Classes**

In [ ]:
class Chess:
    def __init__(self):
        self.row_count = 8
        self.column_count = 8
        self.encoder = ChessEncoder()
        self.action_size = len(self.encoder.index_to_move)

    def __repr__(self):
        return "Chess"

    def get_initial_state(self):
        return ChessState()

    def get_valid_moves(self, state):
        valid_moves = np.zeros(self.action_size, dtype=np.uint8)
        legal_moves = [move.uci() for move in state.board.legal_moves]
        for move in legal_moves:
            idx = self.encoder.move_to_policy_index(move)
            valid_moves[idx] = 1
        return valid_moves

    def get_move_list(self, state):
        move_list = []
        for i, is_valid in enumerate(self.get_valid_moves(state)):
            if is_valid:
                move_list.append(self.encoder.policy_index_to_move(i))
        return move_list

    def get_next_state(self, state, action, player):
        next_state = state.copy()
        move = chess.Move.from_uci(action)
        if move in next_state.board.legal_moves:
            next_state.board.push(move)
        return next_state

    def check_win(self, state, action):
        if action is None:
            return False
        return state.board.is_checkmate()

    def get_value_and_terminated(self, state, action):
        if state.board.is_checkmate():
            return 1, True
        if (state.board.is_stalemate() or
            state.board.is_insufficient_material() or
            state.board.is_fifty_moves() or
            state.board.is_repetition()):
            return 0, True
        return 0, False

    def get_opponent(self, player):
        return -player

    def get_opponent_value(self, value):
        return -value

    def get_encoded_state(self, states):
        encoded_state = np.stack([(states == -value) for value in range(1, 7)] + [(states == value) for value in range(1, 7)]).astype(np.float32)
        if len(states.shape) == 3:
            encoded_state = np.swapaxes(encoded_state, 0, 1)
        return encoded_state

    def board_to_array(self, board):
        state = np.zeros((8, 8), dtype=np.int8)
        piece_map = {'P': 1, 'N': 2, 'B': 3, 'R': 4, 'Q': 5, 'K': 6,  # White pieces
                     'p': -1, 'n': -2, 'b': -3, 'r': -4, 'q': -5, 'k': -6}  # Black pieces
        for square, piece in board.piece_map().items():
            rank = 7 - (square // 8)
            column = square % 8
            state[rank][column] = piece_map[piece.symbol()]
        return state

class ChessState:
    def __init__(self, board=None):
        self.board = board if board else chess.Board()
        self.move_stack = []

    def copy(self):
        new_state = ChessState(self.board.copy())
        new_state.move_stack = self.move_stack.copy()
        return new_state

    def apply_move(self, move_uci):
        move = chess.Move.from_uci(move_uci)
        if move in self.board.legal_moves:
            self.board.push(move)
            self.move_stack.append(move_uci)
            return True
        return False

class ChessEncoder:
    def __init__(self):
        self.move_to_index = {}
        self.index_to_move = []
        self._create_move_mappings()

    def _create_move_mappings(self):
        """Create mappings for all possible moves"""
        files = 'abcdefgh'
        ranks = '12345678'
        pieces = ['q', 'r', 'b', 'n']  # For promotions

        for f1 in files:
            for r1 in ranks:
                for f2 in files:
                    for r2 in ranks:
                        # Regular move
                        self._add_move(f"{f1}{r1}{f2}{r2}")

                        # Promotion moves
                        if r1 == '7' and r2 == '8':  # White promotions
                            for p in pieces:
                                self._add_move(f"{f1}{r1}{f2}{r2}{p}")
                        if r1 == '2' and r2 == '1':  # Black promotions
                            for p in pieces:
                                self._add_move(f"{f1}{r1}{f2}{r2}{p}")

    def _add_move(self, move):
        if move not in self.move_to_index:
            self.move_to_index[move] = len(self.index_to_move)
            self.index_to_move.append(move)

    def move_to_policy_index(self, move_uci):
        return self.move_to_index.get(move_uci, None)

    def policy_index_to_move(self, index):
        return self.index_to_move[index] if index < len(self.index_to_move) else None

In [ ]:
class ChessResNet(nn.Module):
    def __init__(self, game, num_resBlocks, num_hidden, device):
        super().__init__()

        self.device = device

        self.se = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(num_hidden, num_hidden // 16, 1),
            nn.ReLU(),
            nn.Conv2d(num_hidden // 16, num_hidden, 1),
            nn.Sigmoid()
        )

        self.startBlock = nn.Sequential(
            nn.Conv2d(12, num_hidden, kernel_size=3, padding=1),
            nn.BatchNorm2d(num_hidden),
            nn.ReLU()
        )

        self.backBone = nn.ModuleList([ResBlock(num_hidden) for i in range(num_resBlocks)])

        self.policyHead = nn.Sequential(
            nn.Conv2d(num_hidden, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(32 * game.row_count * game.column_count, game.action_size)
        )

        self.valueHead = nn.Sequential(
            nn.Conv2d(num_hidden, 3, kernel_size=3, padding=1),
            nn.BatchNorm2d(3),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(3 * game.row_count * game.column_count, 256),
            nn.ReLU(),
            nn.Linear(256, 1),
            nn.Tanh()
        )

        self.to(device)

    def forward(self, x):
        x = self.startBlock(x)
        for resBlock in self.backBone:
            x = resBlock(x)
            # Apply SE attention
            se_weight = self.se(x)
            x = x * se_weight
        policy = self.policyHead(x)
        value = self.valueHead(x)
        return policy, value

class ResBlock(nn.Module):
    def __init__(self, num_hidden):
        super().__init__()
        self.conv1 = nn.Conv2d(num_hidden, num_hidden, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(num_hidden)
        self.conv2 = nn.Conv2d(num_hidden, num_hidden, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(num_hidden)

    def forward(self, x):
        residual = x
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        x += residual
        x = F.relu(x)
        return x

In [ ]:
class Node:
    def __init__(self, game, args, state, parent=None, action_taken=None, prior=0, visit_count=0):
        self.game = game
        self.args = args
        self.state = state
        self.parent = parent
        self.action_taken = action_taken
        self.prior = prior

        self.children = []

        self.visit_count = visit_count
        self.value_sum = 0

    def is_fully_expanded(self):
        return len(self.children) > 0

    def select(self):
        best_child = None
        best_ucb = -np.inf

        for child in self.children:
            ucb = self.get_ucb(child)
            if ucb > best_ucb:
                best_child = child
                best_ucb = ucb

        return best_child

    def get_ucb(self, child):
        if child.visit_count == 0:
            q_value = 0
        else:
            q_value = 1 - ((child.value_sum / child.visit_count) + 1) / 2
        return q_value + self.args['C'] * (math.sqrt(self.visit_count) / (child.visit_count + 1)) * child.prior

    def expand(self, policy):
        for action, prob in enumerate(policy):
            if prob > 0:
                state = self.state.copy()
                current_action = self.game.encoder.policy_index_to_move(action)
                player = 1 if state.board.turn else -1
                state = self.game.get_next_state(state, current_action, player)
                child = Node(self.game, self.args, state, self, action, prob)
                self.children.append(child)

    def backpropagate(self, value):
        self.value_sum += value
        self.visit_count += 1

        value = self.game.get_opponent_value(value)
        if self.parent is not None:
            self.parent.backpropagate(value)

In [ ]:
class MCTS:
    def __init__(self, game, args, model):
        self.game = game
        self.args = args
        self.model = model

    @torch.no_grad()
    def search(self, state):
        # define root
        root = Node(self.game, self.args, state, visit_count=1)
        encoded_state = self.game.get_encoded_state(self.game.board_to_array(state.board))
        encoded_state = torch.tensor(encoded_state, dtype=torch.float32, device=self.model.device).unsqueeze(0)

        policy, _ = self.model(encoded_state)
        policy = torch.softmax(policy, axis=1).squeeze(0).cpu().numpy()

        valid_moves = self.game.get_valid_moves(state)

        noise = np.random.dirichlet([self.args['dirichlet_alpha']] * self.game.action_size)
        policy = (1 - self.args['dirichlet_epsilon']) * policy + self.args['dirichlet_epsilon'] * noise
        policy *= valid_moves
        policy /= np.sum(policy)

        root.expand(policy)

        for _ in range(self.args['num_searches']):
            node = root

            while node.is_fully_expanded():
                node = node.select() # Selection

            value, is_terminal = self.game.get_value_and_terminated(node.state, node.action_taken)
            value = self.game.get_opponent_value(value)

            if not is_terminal:
                encoded_state = self.game.get_encoded_state(self.game.board_to_array(node.state.board))
                encoded_state = torch.tensor(encoded_state, dtype=torch.float32, device=self.model.device).unsqueeze(0)

                policy, value = self.model(encoded_state)
                policy = torch.softmax(policy, axis=1).squeeze(0).cpu().numpy()

                valid_moves = self.game.get_valid_moves(node.state)
                policy *= valid_moves
                policy /= np.sum(policy)

                value = value.item()

                node.expand(policy) # Expansion

            node.backpropagate(value) # Backpropagation

        action_probs = np.zeros(self.game.action_size) # return visit_counts
        for child in root.children:
            action_probs[child.action_taken] = child.visit_count
        action_probs /= np.sum(action_probs)
        return action_probs

In [ ]:
class AlphaZero:
    def __init__(self, model, optimizer, game, args):
        self.model = model
        self.optimizer = optimizer
        self.game = game
        self.args = args
        self.mcts = MCTS(game, args, model)

    def selfPlay(self):
        memory = []
        player = 1
        state = self.game.get_initial_state()
        move_num = 1

        while True:
            print()
            print(f"Move {move_num}, Player: {player}")
            print(state.board.unicode())
            action_probs = self.mcts.search(state)

            print("Policy probabilities:", action_probs)
            memory.append((state, action_probs, player))

            temperature_action_probs = action_probs ** (1 / self.args['temperature'])
            temperature_action_probs /= np.sum(temperature_action_probs)

            action = np.random.choice(self.game.action_size, p=temperature_action_probs)

            action = self.game.encoder.policy_index_to_move(action)

            print("Chosen move:", action)
            state = self.game.get_next_state(state, action, player)

            print("Board after move:")
            print(state.board.unicode())
            value, is_terminal = self.game.get_value_and_terminated(state, action)

            if is_terminal:
                print(f"Game terminated. {player}")
                print("Final board:")
                print(state.board.unicode())
                returnMemory = []
                for hist_neutral_state, hist_action_probs, hist_player in memory:
                    hist_outcome = value if hist_player == player else self.game.get_opponent_value(value)
                    returnMemory.append((self.game.get_encoded_state(hist_neutral_state),
                                         hist_action_probs,
                                         hist_outcome
                                        ))
                return returnMemory

            player = self.game.get_opponent(player)
            move_num += 1

    def train(self, memory):
        random.shuffle(memory)
        epoch_loss = 0
        for batchIdx in range(0, len(memory), self.args['batch_size']):
            sample = memory[batchIdx:min(len(memory) - 1, batchIdx + self.args['batch_size'])]
            state, policy_targets, value_targets = zip(*sample)

            state, policy_targets, value_targets = np.array(state), np.array(policy_targets), np.array(value_targets).reshape(-1, 1)

            state = torch.tensor(state, dtype=torch.float32, device=self.model.device)
            policy_targets = torch.tensor(policy_targets, dtype=torch.float32, device=self.model.device)
            value_targets = torch.tensor(value_targets, dtype=torch.float32, device=self.model.device)

            out_policy, out_value = self.model(state)

            log_policy = F.log_softmax(out_policy, dim=1)
            policy_loss = -torch.mean(torch.sum(policy_targets * log_policy, dim=1))

            value_loss = F.mse_loss(out_value, value_targets)
            loss = policy_loss + value_loss

            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()
            epoch_loss += loss.item()
        print(f"loss: {epoch_loss/len(sample)}")

    def learn(self):
        for iteration in range(self.args['num_iterations']):
            memory = []

            self.model.eval()
            for selfPlay_iteration in trange(self.args['num_selfPlay_iterations']):
                memory.extend(self.selfPlay())

            self.model.train()
            for epoch in trange(self.args['num_epochs']):
                print(f"Epoch {epoch}", end=" ")
                self.train(memory)

            torch.save(self.model.state_dict(), f"model_{iteration + 1}_{self.game}.pt")
            torch.save(self.optimizer.state_dict(), f"optimizer_{iteration + 1}_{self.game}.pt")

In [ ]:
class MCTSParallel:
    def __init__(self, game, args, model):
        self.game = game
        self.args = args
        self.model = model

    @torch.no_grad()
    def search(self, states, spGames):
        # define root
        encoded_state = self.game.get_encoded_state(np.stack([self.game.board_to_array(state.board) for state in states]))
        encoded_state = torch.tensor(encoded_state, dtype=torch.float32, device=self.model.device)

        policy, _ = self.model(encoded_state)
        policy = torch.softmax(policy, axis=1).cpu().numpy()

        noise = np.random.dirichlet([self.args['dirichlet_alpha']] * self.game.action_size, size=policy.shape[0])
        policy = (1 - self.args['dirichlet_epsilon']) * policy + self.args['dirichlet_epsilon'] * noise

        for i, spg in enumerate(spGames):
            spg_policy = policy[i]
            valid_moves = self.game.get_valid_moves(states[i])
            spg_policy *= valid_moves
            spg_policy /= np.sum(spg_policy)

            spg.root = Node(self.game, self.args, states[i], visit_count=1)
            spg.root.expand(spg_policy)

        for _ in range(self.args['num_searches']):
            for spg in spGames:
                spg.node = None
                node = spg.root

                while node.is_fully_expanded():
                    node = node.select() # Selection

                value, is_terminal = self.game.get_value_and_terminated(node.state, node.action_taken)
                value = self.game.get_opponent_value(value)

                if is_terminal:
                    node.backpropagate(value) # Backpropagation
                else:
                    spg.node = node

            expandable_spGames = [mappingIdx for mappingIdx in range(len(spGames)) if spGames[mappingIdx].node is not None]
            if len(expandable_spGames) > 0:
                states = np.stack([self.game.board_to_array(spGames[mappingIdx].node.state.board) for mappingIdx in expandable_spGames])
                encoded_state = self.game.get_encoded_state(states)
                encoded_state = torch.tensor(encoded_state, dtype=torch.float32, device=self.model.device)

                policy, value = self.model(encoded_state)
                policy = torch.softmax(policy, axis=1).cpu().numpy()
                value = value.cpu().numpy()

            for i, mappingIdx in enumerate(expandable_spGames):
                node = spGames[mappingIdx].node
                spg_policy, spg_value = policy[i], value[i]

                valid_moves = self.game.get_valid_moves(node.state)
                spg_policy *= valid_moves
                spg_policy /= np.sum(spg_policy)

                node.expand(spg_policy) # Expansion
                node.backpropagate(spg_value) # Backpropagation

In [ ]:
class AlphaZeroParallel:
    def __init__(self, model, optimizer, game, args):
        self.model = model
        self.optimizer = optimizer
        self.game = game
        self.args = args
        self.mcts = MCTSParallel(game, args, model)

    def selfPlay(self):
        return_memory = []
        player = 1
        spGames = [SPG(self.game) for _ in range(self.args['num_parallel_games'])]

        while len(spGames) > 0:
            states = np.stack([spg.state for spg in spGames])
            self.mcts.search(states, spGames)

            for i in range(len(spGames))[::-1]:
                spg = spGames[i]

                action_probs = np.zeros(self.game.action_size) # return visit_counts
                for child in spg.root.children:
                    action_probs[child.action_taken] = child.visit_count
                action_probs /= np.sum(action_probs)

                spg.memory.append((spg.root.state, action_probs, player))

                temperature_action_probs = action_probs ** (1 / self.args['temperature'])
                temperature_action_probs /= np.sum(temperature_action_probs)

                action = np.random.choice(self.game.action_size, p=temperature_action_probs)

                action = self.game.encoder.policy_index_to_move(action)

                spg.state = self.game.get_next_state(spg.state, action, player)

                value, is_terminal = self.game.get_value_and_terminated(spg.state, action)

                if is_terminal:
                    for hist_neutral_state, hist_action_probs, hist_player in spg.memory:
                        hist_outcome = value if hist_player == player else self.game.get_opponent_value(value)
                        return_memory.append((self.game.get_encoded_state(hist_neutral_state),
                                            hist_action_probs,
                                            hist_outcome))
                    del spGames[i]
            player = self.game.get_opponent(player)

        return return_memory

    def train(self, memory):
        random.shuffle(memory)
        epoch_loss = 0
        for batchIdx in range(0, len(memory), self.args['batch_size']):
            sample = memory[batchIdx:min(len(memory) - 1, batchIdx + self.args['batch_size'])]
            state, policy_targets, value_targets = zip(*sample)

            state, policy_targets, value_targets = np.array(state), np.array(policy_targets), np.array(value_targets).reshape(-1, 1)

            state = torch.tensor(state, dtype=torch.float32, device=self.model.device)
            policy_targets = torch.tensor(policy_targets, dtype=torch.float32, device=self.model.device)
            value_targets = torch.tensor(value_targets, dtype=torch.float32, device=self.model.device)

            out_policy, out_value = self.model(state)

            log_policy = F.log_softmax(out_policy, dim=1)
            policy_loss = -torch.mean(torch.sum(policy_targets * log_policy, dim=1))

            value_loss = F.mse_loss(out_value, value_targets)
            loss = policy_loss + value_loss

            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()
            epoch_loss += loss.item()
        print(f"loss: {epoch_loss/len(sample)}")

    def learn(self):
        for iteration in range(self.args['num_iterations']):
            memory = []

            self.model.eval()
            for selfPlay_iteration in trange(self.args['num_selfPlay_iterations'] // self.args['num_parallel_games']):
                memory.extend(self.selfPlay())

            self.model.train()
            for epoch in trange(self.args['num_epochs']):
                print(f"Epoch {epoch + 1}", end=" ")
                self.train(memory)

            torch.save(self.model.state_dict(), f"model_{iteration + 1}_{self.game}.pt")
            torch.save(self.optimizer.state_dict(), f"optimizer_{iteration + 1}_{self.game}.pt")

class SPG:
    def __init__(self, game):
        self.state = game.get_initial_state()
        self.memory = []
        self.root = None
        self.node = None

# **Train AI Model**
Time Duration:

*   CPU: 62 hrs
*   GPU: 5 hrs


In [ ]:
game = Chess()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = ChessResNet(game, 19, 256, device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=0.0001)

args = {'C': 2,
        'num_searches': 800,
        'num_iterations': 20,
        'num_selfPlay_iterations': 1000,
        'num_parallel_games': 1000, #Default: 200
        'num_epochs': 6,
        'batch_size': 256,
        'temperature': 1.25,
        'dirichlet_epsilon': 0.25,
        'dirichlet_alpha': 0.3
        }

alphazero = AlphaZero(model, optimizer, game, args)
alphazero.learn()

  0%|          | 0/1000 [00:00<?, ?it/s]


Move 1, Player: 1
♜ ♞ ♝ ♛ ♚ ♝ ♞ ♜
♟ ♟ ♟ ♟ ♟ ♟ ♟ ♟
⭘ ⭘ ⭘ ⭘ ⭘ ⭘ ⭘ ⭘
⭘ ⭘ ⭘ ⭘ ⭘ ⭘ ⭘ ⭘
⭘ ⭘ ⭘ ⭘ ⭘ ⭘ ⭘ ⭘
⭘ ⭘ ⭘ ⭘ ⭘ ⭘ ⭘ ⭘
♙ ♙ ♙ ♙ ♙ ♙ ♙ ♙
♖ ♘ ♗ ♕ ♔ ♗ ♘ ♖
Policy probabilities: [0.045   0.04    0.05875 ... 0.      0.      0.     ]
Chosen move: d2d4
Board after move:
♜ ♞ ♝ ♛ ♚ ♝ ♞ ♜
♟ ♟ ♟ ♟ ♟ ♟ ♟ ♟
⭘ ⭘ ⭘ ⭘ ⭘ ⭘ ⭘ ⭘
⭘ ⭘ ⭘ ⭘ ⭘ ⭘ ⭘ ⭘
⭘ ⭘ ⭘ ♙ ⭘ ⭘ ⭘ ⭘
⭘ ⭘ ⭘ ⭘ ⭘ ⭘ ⭘ ⭘
♙ ♙ ♙ ⭘ ♙ ♙ ♙ ♙
♖ ♘ ♗ ♕ ♔ ♗ ♘ ♖

Move 2, Player: -1
♜ ♞ ♝ ♛ ♚ ♝ ♞ ♜
♟ ♟ ♟ ♟ ♟ ♟ ♟ ♟
⭘ ⭘ ⭘ ⭘ ⭘ ⭘ ⭘ ⭘
⭘ ⭘ ⭘ ⭘ ⭘ ⭘ ⭘ ⭘
⭘ ⭘ ⭘ ♙ ⭘ ⭘ ⭘ ⭘
⭘ ⭘ ⭘ ⭘ ⭘ ⭘ ⭘ ⭘
♙ ♙ ♙ ⭘ ♙ ♙ ♙ ♙
♖ ♘ ♗ ♕ ♔ ♗ ♘ ♖
Policy probabilities: [0. 0. 0. ... 0. 0. 0.]
Chosen move: b7b5
Board after move:
♜ ♞ ♝ ♛ ♚ ♝ ♞ ♜
♟ ⭘ ♟ ♟ ♟ ♟ ♟ ♟
⭘ ⭘ ⭘ ⭘ ⭘ ⭘ ⭘ ⭘
⭘ ♟ ⭘ ⭘ ⭘ ⭘ ⭘ ⭘
⭘ ⭘ ⭘ ♙ ⭘ ⭘ ⭘ ⭘
⭘ ⭘ ⭘ ⭘ ⭘ ⭘ ⭘ ⭘
♙ ♙ ♙ ⭘ ♙ ♙ ♙ ♙
♖ ♘ ♗ ♕ ♔ ♗ ♘ ♖

Move 3, Player: 1
♜ ♞ ♝ ♛ ♚ ♝ ♞ ♜
♟ ⭘ ♟ ♟ ♟ ♟ ♟ ♟
⭘ ⭘ ⭘ ⭘ ⭘ ⭘ ⭘ ⭘
⭘ ♟ ⭘ ⭘ ⭘ ⭘ ⭘ ⭘
⭘ ⭘ ⭘ ♙ ⭘ ⭘ ⭘ ⭘
⭘ ⭘ ⭘ ⭘ ⭘ ⭘ ⭘ ⭘
♙ ♙ ♙ ⭘ ♙ ♙ ♙ ♙
♖ ♘ ♗ ♕ ♔ ♗ ♘ ♖
Policy probabilities: [0.05125 0.02625 0.02625 ... 0.      0.      0.     ]
Chosen move: b2b3
Board after mo

KeyboardInterrupt: 